# hybrid_r1_5m alone - Kaggle TPU VM (learner on the TPU, all CPU cores as rollout workers)
Commit `c85788a` of Traffic-Light-ABM. One case only (`ONLY=hybrid_r1_5m`), so the whole VM (nproc - 4 workers) serves this run. Resumes from `hybrid_r1_5m.tar` in /kaggle/input (Input = notebookb5373f1a91 pinned to Version 1; later sessions: this notebook's own latest output). Cell 1 aborts if that tar is missing, so a wrong Input never starts a fresh run.


In [ ]:
import os, subprocess, sys, shutil, time
COMMIT = "c85788a7d6a13012ac7300abb10f6314cae7ddc1"
REPO = "https://github.com/ngocbityta/Traffic-Light-ABM"
WORK = "/tmp/work"
SRC = f"{WORK}/gama"
os.makedirs(WORK, exist_ok=True)
open(f"{WORK}/T0", "w").write(str(time.time()))   # session start, for the 9h budget

def sh(cmd, check=True):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True, executable="/bin/bash")
    if check and r.returncode:
        raise SystemExit(f"command failed ({r.returncode}): {cmd}")

sh("nproc; free -g; df -h /tmp /kaggle/working | cat; python -V; lscpu | grep -E 'Model name|^CPU\\(s\\)|Thread|Socket'")
sh("ls -laR /kaggle/input 2>/dev/null | head -40", check=False)
if not os.path.isdir(SRC):
    sh(f"git clone -q {REPO} {SRC}")
sh(f"cd {SRC} && git checkout -q {COMMIT} && git log --oneline -1")
import glob
assert glob.glob("/kaggle/input/**/hybrid_r1_5m.tar", recursive=True), \
    "no hybrid_r1_5m.tar under /kaggle/input: wrong Input version, refusing to start a fresh run"


In [ ]:
VENV = f"{WORK}/venv"
PY = f"{VENV}/bin/python"
if not os.path.exists(PY):
    sh("pip install -q uv")
    sh(f"uv venv --python 3.11 {VENV}")
    sh(f"uv pip install --python {PY} 'torch==2.7.0' 'torch_xla[tpu]==2.7.0' -f https://storage.googleapis.com/libtpu-releases/index.html")
    sh(f"uv pip install --python {PY} 'gama-client' 'numpy<2' pandas matplotlib 'Cython==3.3.1' setuptools pytest")
sh(f"cd {SRC} && {PY} python-simulation/scripts/build_native.py 2>&1 | tail -3")
out = subprocess.run(
    [PY, "-c", "import sys; sys.path.insert(0,'python-simulation/simulation'); "
               "import python_simulation.python_sim as p; print(p.__file__)"],
    cwd=SRC, capture_output=True, text=True).stdout.strip()
print("sim module:", out)
assert out.endswith(".so"), "sim is NOT the Cython build - would be very slow"
sh(f"{PY} -c \"import torch;print('torch',torch.__version__)\"")
sh("ldd --version | head -1", check=False)

# glibc >= 2.41 refuses to dlopen a .so that asks for an executable stack
# ("_XLAC...so: cannot enable executable stack as shared object requires: Invalid argument",
# Version 2 failed on it). Clear that ELF flag (same as `patchelf --clear-execstack`).
import struct, glob
def clear_execstack(path):
    with open(path, "r+b") as f:
        ident = f.read(64)
        if ident[:4] != b"\x7fELF" or ident[4] != 2 or ident[5] != 1:
            return False
        phoff = struct.unpack_from("<Q", ident, 0x20)[0]
        phentsize, phnum = struct.unpack_from("<HH", ident, 0x36)
        for i in range(phnum):
            off = phoff + i * phentsize
            f.seek(off)
            p_type, p_flags = struct.unpack("<II", f.read(8))
            if p_type == 0x6474E551 and p_flags & 1:
                f.seek(off + 4)
                f.write(struct.pack("<I", p_flags & ~1))
                return True
    return False
site = glob.glob(f"{VENV}/lib/python3*/site-packages")[0]
fixed = [p for p in glob.glob(f"{site}/**/*.so*", recursive=True)
         if os.path.isfile(p) and not os.path.islink(p) and clear_execstack(p)]
print("execstack cleared:", [os.path.relpath(p, site) for p in fixed], flush=True)

# informative only: the launcher probes each chip again and falls back to --device cpu
# for a run whose chip fails, so a broken TPU never wastes the session
sh(f'''PJRT_DEVICE=TPU {PY} -c "import torch, torch_xla; d=torch_xla.device(); x=torch.randn(64,64,device=d); print('xla device', d, float((x@x).sum()))" ''', check=False)

In [ ]:
sh(f"cd {SRC} && timeout 900 {PY} -m pytest -q -x train/tests 2>&1 | tail -8", check=False)

In [ ]:
import os
os.environ["XLA"] = "1"
os.environ["ONLY"] = "hybrid_r1_5m"

# 5 train.py runs in parallel on one Kaggle TPU VM (CPU only, TPU unused).
# Each run: own output dir under /tmp/work/runs, own backup tar in
# /kaggle/working, auto-resume when /kaggle/input holds <name>.tar from an
# earlier version of this notebook.
import glob, os, shutil, subprocess, tarfile, time

WORK = os.environ.get("WORK", "/tmp/work")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
INPUT = os.environ.get("INPUT_DIR", "/kaggle/input")
SRC = f"{WORK}/gama"
PY = f"{WORK}/venv/bin/python"
SMOKE = os.environ.get("SMOKE") == "1"           # local test only
# XLA=1: learner of every run on a TPU chip (torch_xla), workers/validation on CPU.
# Needs a train.py with --device xla; --allow-code-change lets old CPU-run tars resume.
XLA = os.environ.get("XLA") == "1"

RUNS = [  # name, algorithm, reward mode, extra train.py args
    ("rainbow_r1_5m", "rainbow", "r1", []),
    ("rainbow_r2_5m", "rainbow", "r2", []),
    ("rainbow_r4_5m", "rainbow", "r4", []),
    ("hybrid_r1_5m", "hybrid", "r1", []),
    # second, independent RainbowDQN R4 run: other training seeds / init,
    # same validation seeds (10_000_001..3) so it compares with rainbow_r4_5m
    ("rainbow_r4_seed2_5m", "rainbow", "r4", ["--seed-base", "20000000"]),
]
if os.environ.get("ONLY"):
    RUNS = [r for r in RUNS if r[0] in os.environ["ONLY"].split(",")]
# CPU-notebook branch: same training settings, own names (cpu_<run>)
PREFIX = os.environ.get("RUN_PREFIX", "")
RUNS = [(PREFIX + name, *rest) for name, *rest in RUNS]

SESSION = float(os.environ.get("SESSION_HOURS", 9)) * 3600   # TPU VM 9h, CPU 12h
RESERVE = 1800                                   # after train: last validation, save, sync, upload
try:
    T0 = float(open(f"{WORK}/T0").read())        # written by cell 1
except Exception:
    T0 = time.time()
NPROC = os.cpu_count()
PER = NPROC // len(RUNS)
if SMOKE:
    WORKERS, LEARNER_THREADS = 1, 1
elif PER >= 8:                                   # 3 validation procs + learner threads
    WORKERS, LEARNER_THREADS = max(1, PER - 4), 2
else:                                            # 4-core CPU notebook
    WORKERS, LEARNER_THREADS = max(1, PER - 2), 1
BUDGET = int(os.environ.get("TRAIN_SECONDS", SESSION - RESERVE - (time.time() - T0)))
HARD_STOP = T0 + SESSION - 900                   # kill stragglers, keep 15 min for sync
MONITOR_EVERY = 20 if SMOKE else 1800
SYNC_EVERY = 10 if SMOKE else 900
SMOKE_ARGS = ("--total-steps 300 --episode-horizon 600 --warmup 40 "
              "--validate-every 4 --checkpoint-every 2") if SMOKE else ""
print(f"nproc {NPROC} | runs {len(RUNS)} | workers/run {WORKERS} | "
      f"learner threads/run {LEARNER_THREADS} | train budget {BUDGET}s ({BUDGET/3600:.2f}h)",
      flush=True)
assert SMOKE or BUDGET > 600, "not enough session time left"


def tail(path, n=1, grep=None):
    try:
        lines = open(path, errors="replace").read().splitlines()
    except Exception:
        return ""
    if grep:  # str or tuple; async runs print "[Rollout ...", single-worker runs "[Ep ..."
        keys = (grep,) if isinstance(grep, str) else grep
        lines = [l for l in lines if any(k in l for k in keys)]
    return "\n".join(lines[-n:])


def restore(name, run):
    """Resume source: a run dir from this session, else a tar from /kaggle/input."""
    if os.path.exists(f"{run}/training_state.pt"):
        return True
    hits = sorted(glob.glob(f"{INPUT}/**/{name}.tar", recursive=True), key=os.path.getmtime)
    if not hits:
        return False
    if len(hits) > 1:
        print(f"[{name}] several backups, using newest: {hits}", flush=True)
    os.makedirs(run, exist_ok=True)
    with tarfile.open(hits[-1]) as archive:
        try:
            archive.extractall(run, filter="data")
        except TypeError:                        # Python without tar filters
            archive.extractall(run)
    # keep the restored state in this version's output until the first sync
    shutil.copy(hits[-1], f"{OUT}/{name}.tar")
    print(f"[{name}] restored from {hits[-1]}", flush=True)
    return True


env = dict(os.environ, OMP_NUM_THREADS="1", MKL_NUM_THREADS="1", PYTHONUNBUFFERED="1")


def tpu_env(index):
    """One TPU chip per run: several processes cannot share the chips, so give
    each run its own chip (v5e-8 has 8) through the libtpu process variables."""
    if not XLA:
        return {}
    port = 8476 + index
    return dict(PJRT_DEVICE="TPU", TPU_VISIBLE_CHIPS=str(index),
                TPU_CHIPS_PER_PROCESS_BOUNDS="1,1,1", TPU_PROCESS_BOUNDS="1,1,1",
                TPU_PROCESS_PORT=str(port), TPU_PROCESS_ADDRESSES=f"localhost:{port}",
                CLOUD_TPU_TASK_ID="0")
os.makedirs(OUT, exist_ok=True)


def probe_tpu():
    """All runs open their own chip at once for ~25 s; a run whose chip fails
    trains on CPU instead (a VM with at least one run on the TPU is not idle)."""
    code = ("import time,torch,torch_xla;d=torch_xla.device();x=torch.randn(64,64,device=d);"
            "print('XLA_OK',d,float((x@x).sum()),flush=True);time.sleep(25)")
    procs = [subprocess.Popen([PY, "-c", code], env=dict(env, **tpu_env(i)),
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
             for i in range(len(RUNS))]
    ok = []
    for i, proc in enumerate(procs):
        try:
            out = proc.communicate(timeout=240)[0]
        except subprocess.TimeoutExpired:
            proc.kill()
            out = proc.communicate()[0]
        good = "XLA_OK" in out
        ok.append(good)
        print(f"[tpu probe {i}] {'ok' if good else 'FAILED'}: {out.strip()[-300:]}", flush=True)
    return ok


TPU_OK = probe_tpu() if XLA else [False] * len(RUNS)
print(f"TPU learner for runs: {[r[0] for r, good in zip(RUNS, TPU_OK) if good]}", flush=True)
jobs = []
for index, (name, algorithm, reward, extra) in enumerate(RUNS):
    run = f"{WORK}/runs/{name}"
    flag = f"{WORK}/DONE_{name}"
    if os.path.exists(flag):
        os.remove(flag)
    resume = restore(name, run)
    DEVICE_ARGS = ("--device xla" if TPU_OK[index] else "--device cpu") + \
                  (" --allow-code-change" if XLA else "")
    os.makedirs(run, exist_ok=True)
    script = f"""set -u
cd {SRC}
source train/colab_run/run_config.sh
exec {PY} -u -m train.logic.train "${{TRAIN_ARGS[@]}}" \\
  --algorithm {algorithm} --reward-mode {reward} {' '.join(extra)} \\
  {DEVICE_ARGS} --workers {WORKERS} --learner-threads {LEARNER_THREADS} \\
  --checkpoint-every 200 --max-train-seconds {BUDGET} \\
  {f'--resume-dir {run}' if resume else ''} --output-dir {run} {SMOKE_ARGS}
"""
    open(f"{run}.sh", "w").write(script)
    console = open(f"{run}/console.log", "a")
    train = subprocess.Popen(["bash", f"{run}.sh"], stdout=console,
                             stderr=subprocess.STDOUT, env=dict(env, **(tpu_env(index) if TPU_OK[index] else {})),
                             start_new_session=True)
    sync = subprocess.Popen([PY, f"{SRC}/train/colab_run/sync_run.py", "--src", run,
                             "--tar", f"{OUT}/{name}.tar", "--every", str(SYNC_EVERY),
                             "--until", flag],
                            stdout=open(f"{run}.sync.log", "a"), stderr=subprocess.STDOUT)
    jobs.append(dict(name=name, run=run, flag=flag, train=train, sync=sync,
                     resume=resume, last=(time.time(), None)))
    print(f"[{name}] started {algorithm} {reward} {' '.join(extra)} "
          f"{'RESUME' if resume else 'FRESH'} pid {train.pid}", flush=True)
    time.sleep(2 if SMOKE else 20)               # stagger imports / process spawns


def report():
    print(f"\n=== {(time.time() - T0) / 3600:.2f} h since session start ===", flush=True)
    for job in jobs:
        log = f"{job['run']}/console.log"
        state = "running" if job["train"].poll() is None else f"exit {job['train'].returncode}"
        line = tail(log, grep=("[Rollout ", "[Ep "))
        rate = ""
        try:
            step = int(line.split("step ")[1].split("/")[0])
            then, before = job["last"]
            if before is not None:
                rate = f" | {(step - before) / (time.time() - then):.1f} dec/s"
            job["last"] = (time.time(), step)
        except Exception:
            pass
        print(f"[{job['name']}] {state}{rate}\n  {line[:150]}\n  "
              f"{tail(log, grep='[Validation]')[:150]}\n  "
              f"{tail(job['run'] + '.sync.log')[:150]}", flush=True)
    subprocess.run(f"free -g | head -2; df -h /tmp {OUT} | cat; du -sh {OUT}",
                   shell=True, executable="/bin/bash")


next_report = time.time() + MONITOR_EVERY
while any(job["train"].poll() is None for job in jobs):
    time.sleep(5)
    if time.time() >= next_report:
        report()
        next_report += MONITOR_EVERY
    if time.time() > HARD_STOP:
        for signal in (15, 9):
            for job in jobs:
                if job["train"].poll() is None:
                    print(f"[{job['name']}] past hard stop, signal {signal}", flush=True)
                    try:
                        os.killpg(job["train"].pid, signal)
                    except ProcessLookupError:
                        pass
            time.sleep(60)

report()
for job in jobs:
    open(job["flag"], "w").close()               # syncs do one final copy, then exit
for job in jobs:
    try:
        job["sync"].wait(timeout=1200)
    except subprocess.TimeoutExpired:
        job["sync"].kill()
    shutil.copy(f"{job['run']}/console.log", f"{OUT}/{job['name']}.console.log")
    print(f"\n##### {job['name']} exit {job['train'].returncode}\n"
          f"{tail(job['run'] + '/console.log', 6)}\n{tail(job['run'] + '.sync.log', 2)}",
          flush=True)
subprocess.run(f"ls -la {OUT}; du -sh {OUT}", shell=True, executable="/bin/bash")